# Equity Regime Colour：把回测 Equity 曲线压缩成一维策略环境色带

回测 equity 曲线 $E_t$ 是二维图：横轴为时间、纵轴为资金。这里要发明的不是另一条 equity 曲线，而是每一个时点只有一个数、可用一种颜色表示的 **策略环境分数** $C_t$。

用途是比较：把不同策略的色带上下排列后，一眼看出某段市场中哪些策略有较稳定的正向表现，哪些策略处于不适合交易的阶段。它是研究和筛选指标，不是未来收益预测或自动开关策略的许可。

## 1. 为什么不能直接给 $\Delta E_t$ 上色？

直接使用 $E_t-E_{t-1}$ 有三个问题：

- 资金从 1 万变成 100 万后，同一策略的数值会放大 100 倍；
- 正收益但剧烈波动的区间，未必比平稳正收益更适合执行；
- 盈利后的权益高点会让绝对金额变大，造成“越赚钱、颜色越绿”的机械偏差。

因此基本输入应是无量纲的对数收益：

$$r_t=\log(E_t/E_{t-1}).$$

若 equity 可为零或负数，先改用逐笔/逐 bar 的净值收益，或停止在该段定义此指标；不能对非正数取对数。

## 2. 最小候选：纯变化率颜色

给定一个短观察窗 $h$，最简单的一维指标是：

$$G_t^{(h)}=\frac{1}{h}\sum_{i=0}^{h-1}r_{t-i}=\frac{\log(E_t/E_{t-h})}{h}. $$

令绿色表示 $G_t^{(h)}>0$，红色表示 $G_t^{(h)}<0$，饱和度表示 $|G_t^{(h)}|$。这严格回答“颜色表示 equity 的变化率”，并且可跨不同初始资金比较。

但它不惩罚风险：一段靠少数大赚 bar 得到的绿色，和一段稳定的小幅盈利没有区别。所以它应作为基线，而不是最终的“适合交易”定义。

## 3. 建议的第一版定义：因果 Equity Regime Colour（ERC）

取短窗 $h$（反应速度）、风险窗 $w$（通常 $w\ge h$）、回撤惩罚尺度 $d_0>0$。在 $t$ 时刻、只使用截至 $t$ 的信息：

$$\mu_t=\operatorname{mean}(r_{t-h+1},\ldots,r_t),\qquad\sigma_t=\operatorname{std}(r_{t-w+1},\ldots,r_t),$$

$$D_t=1-\frac{E_t}{\max_{u\le t}E_u},$$

$$Q_t=\frac{\sqrt{h}\,\mu_t}{\sigma_t+\varepsilon},\qquad C_t=\tanh\!\left(\frac{Q_t}{q_0}\right)\exp\!\left(-\frac{D_t}{d_0}\right).$$

其中 $\varepsilon$ 防止零波动除法，$q_0$ 决定颜色多快饱和。$C_t\in[-1,1]$：

- 深绿（接近 $+1$）：近期正对数增长相对波动强，且接近 equity 高点；
- 灰/黄（接近 $0$）：没有足够的风险调整后优势，或虽有优势但正处于明显回撤；
- 深红（接近 $-1$）：近期风险调整后增长为负。

$Q_t$ 类似滚动 Sharpe 的有符号强度；$D_t$ 明确表达“策略目前正在从自身高点回撤”。这两个部分分别回答 **是否有优势** 与 **是否正承受路径风险**。

## 4. 颜色本身是一维曲线

画法不再是 $E_t$ 的纵轴曲线，而是一条水平色带：时间仍是横轴，第 $t$ 个窄格的颜色只由 $C_t$ 决定。比较 $m$ 个策略时，每个策略一行色带；每行依然只有一维时间序列。

为避免比较时偷看未来：

1. 每个 $C_t$ 只允许使用 $r_{\le t}$ 与 $E_{\le t}$；
2. 所有策略使用同一 $h,w,d_0,q_0$，或使用事前冻结的换算规则；
3. 若用分位数确定颜色阈值，只能用训练期分位数固定后应用到测试期。

色带适合做诊断。若把它用于“策略只在绿色时交易”，则必须在独立样本上回测，因为这已经是一个新的择时策略。

In [ ]:
import numpy as np
import pandas as pd
import pymysql
from time import perf_counter
import re
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

BETA = 0.34
BARS_PER_TRADING_DAY = 79
SIGNAL_WINDOW = 20 * BARS_PER_TRADING_DAY
DB_CONFIG = {'host': 'localhost', 'user': 'quant', 'password': '1q2w3e', 'database': 'quant_data'}

def load_sblk_bhp_5m():
    """All common SBLK/BHP 5-minute bars; no date restriction or resampling."""
    query = """
        SELECT a.time_utc, a.price AS sblk, b.price AS bhp
        FROM us_market_prices a FORCE INDEX (idx_us_market_prices_ticker_time_price)
        JOIN us_market_prices b FORCE INDEX (idx_us_market_prices_ticker_time_price)
          ON b.ticker = 'BHP' AND b.time_utc = a.time_utc
        WHERE a.ticker = 'SBLK'
        ORDER BY a.time_utc
    """
    print('Loading all common SBLK/BHP 5-minute bars from MySQL…', flush=True)
    started_at = perf_counter()
    conn = pymysql.connect(**DB_CONFIG, connect_timeout=10, read_timeout=60, write_timeout=60)
    try:
        data = pd.read_sql(query, conn)
    finally:
        conn.close()
    print(f'Loaded {len(data):,} joined rows in {perf_counter() - started_at:.1f}s.', flush=True)
    data['time_utc'] = pd.to_datetime(data['time_utc'], utc=True)
    data['time'] = data['time_utc'].dt.tz_localize(None)
    data['time_bj'] = data['time'] + pd.Timedelta(hours=8)
    return data.dropna().drop_duplicates('time_utc').reset_index(drop=True)

def equity_regime_colour(equity, growth_window=20 * BARS_PER_TRADING_DAY, risk_window=60 * BARS_PER_TRADING_DAY, drawdown_scale=0.10, saturation=1.0, eps=1e-12):
    """Causal ERC score. equity must be a positive pandas Series."""
    equity = pd.Series(equity, dtype=float)
    if (equity <= 0).any():
        raise ValueError('Equity must stay positive to use log returns.')
    returns = np.log(equity).diff()
    mean_return = returns.rolling(growth_window, min_periods=growth_window).mean()
    volatility = returns.rolling(risk_window, min_periods=risk_window).std(ddof=0)
    drawdown = 1.0 - equity / equity.cummax()
    quality = np.sqrt(growth_window) * mean_return / (volatility + eps)
    colour = np.tanh(quality / saturation) * np.exp(-drawdown / drawdown_scale)
    return pd.DataFrame({'equity': equity, 'log_return': returns, 'drawdown': drawdown, 'quality': quality, 'erc': colour})

def batteries_lab_equity(data, beta=BETA, signal_window=SIGNAL_WINDOW, initial_equity=10_000.0):
    """The Batteries Lab 5-minute rules: re-entry, trailing -2σ stop, z(0) take-profit."""
    result = data.copy()
    result['spread'] = result['sblk'] - beta * result['bhp']
    result['spread_mean'] = result['spread'].rolling(signal_window, min_periods=signal_window).mean()
    result['spread_sigma'] = result['spread'].rolling(signal_window, min_periods=signal_window).std(ddof=0)
    result['zscore'] = (result['spread'] - result['spread_mean']) / result['spread_sigma'].where(result['spread_sigma'] > 0)
    sessions, _ = pd.factorize(result['time'].dt.normalize(), sort=False)
    result['plot_x'] = np.arange(len(result))
    result['trading_day_number'] = sessions

    trades, open_trade, previous_z = [], None, np.nan
    for _, row in result.iterrows():
        zscore = row['zscore']
        if not np.isfinite(zscore):
            continue
        if open_trade is None:
            inside = -2.0 < zscore < 2.0
            if previous_z >= 2.0 and inside:
                side = 'Short spread'
            elif previous_z <= -2.0 and inside:
                side = 'Long spread'
            else:
                previous_z = zscore
                continue
            gross = float(row['sblk']) + abs(beta) * float(row['bhp'])
            sigma_pct = 0.0 if gross <= 0 else float(row['spread_sigma']) / gross * 100.0
            open_trade = {'side': side, 'entry_x': int(row['plot_x']), 'entry_day': int(row['trading_day_number']), 'entry_spread': float(row['spread']), 'entry_y': float(row['sblk']), 'entry_bhp': float(row['bhp']), 'gross': gross, 'sigma_pct': sigma_pct, 'peak_return_pct': 0.0}
            previous_z = zscore
            continue
        direction = 1.0 if open_trade['side'] == 'Long spread' else -1.0
        pnl = direction * (float(row['spread']) - open_trade['entry_spread'])
        return_pct = 100.0 * pnl / open_trade['gross'] if open_trade['gross'] > 0 else 0.0
        open_trade['peak_return_pct'] = max(open_trade['peak_return_pct'], return_pct, 0.0)
        can_exit = int(row['trading_day_number']) > open_trade['entry_day']
        stop_level = open_trade['peak_return_pct'] - 2.0 * open_trade['sigma_pct']
        take_profit = (direction > 0 and zscore >= 0.0) or (direction < 0 and zscore <= 0.0)
        if can_exit and (return_pct <= stop_level or take_profit):
            open_trade.update({'exit_x': int(row['plot_x']), 'exit_spread': float(row['spread'])})
            trades.append(open_trade)
            open_trade = None
        previous_z = zscore
    if open_trade is not None:
        row = result.iloc[-1]
        open_trade.update({'exit_x': int(row['plot_x']), 'exit_spread': float(row['spread'])})
        trades.append(open_trade)

    result['equity'] = initial_equity
    capital = initial_equity
    for trade in trades:
        mask = result['plot_x'].between(trade['entry_x'], trade['exit_x'])
        direction = 1.0 if trade['side'] == 'Long spread' else -1.0
        pnl = direction * ((result.loc[mask, 'sblk'] - trade['entry_y']) - beta * (result.loc[mask, 'bhp'] - trade['entry_bhp']))
        result.loc[mask, 'equity'] = capital * (1.0 + pnl / trade['gross'])
        capital = float(result.loc[mask, 'equity'].iloc[-1])
        result.loc[result['plot_x'] > trade['exit_x'], 'equity'] = capital
    return result, trades

BASELINE_STRATEGY_NAME = 'Base Strategy: Immediate ±2σ Entry / Zero Exit'

def base_strategy_equity(data, beta=BETA, signal_window=SIGNAL_WINDOW, initial_equity=10_000.0):
    """Touch ±2σ to enter; exit at z=0; no stop or take-profit; next-bar execution."""
    result = data.copy()
    result['spread'] = result['sblk'] - beta * result['bhp']
    mean = result['spread'].rolling(signal_window, min_periods=signal_window).mean()
    sigma = result['spread'].rolling(signal_window, min_periods=signal_window).std(ddof=0)
    result['zscore'] = (result['spread'] - mean) / sigma.where(sigma > 0)
    position = np.zeros(len(result), dtype=float)
    current_position, entries = 0.0, 0
    for index, zscore in enumerate(result['zscore']):
        if not np.isfinite(zscore):
            current_position = 0.0
        elif current_position == 0.0 and zscore <= -2.0:
            current_position, entries = 1.0, entries + 1
        elif current_position == 0.0 and zscore >= 2.0:
            current_position, entries = -1.0, entries + 1
        elif current_position > 0.0 and zscore >= 0.0:
            current_position = 0.0
        elif current_position < 0.0 and zscore <= 0.0:
            current_position = 0.0
        position[index] = current_position
    gross_notional = result['sblk'].shift(1) + abs(beta) * result['bhp'].shift(1)
    spread_return = result['spread'].diff() / gross_notional
    result['strategy_return'] = pd.Series(position, index=result.index).shift(1).fillna(0.0) * spread_return.fillna(0.0)
    result['equity'] = initial_equity * (1.0 + result['strategy_return']).cumprod()
    return result, entries

BETA_GRID = np.round(np.arange(0.1, 0.61, 0.1), 1)
bars = load_sblk_bhp_5m()
results_by_beta = {}
for beta in BETA_GRID:
    backtest, entry_count = base_strategy_equity(bars, beta=beta)
    regime = equity_regime_colour(backtest['equity'])
    regime.index = backtest['time_utc']
    results_by_beta[float(beta)] = {'regime': regime, 'entries': entry_count}
print(f'{len(bars):,} common 5-minute bars: {bars.time_utc.iloc[0]} to {bars.time_utc.iloc[-1]}')
for beta, item in results_by_beta.items():
    equity = item['regime']['equity']
    total_return = equity.iloc[-1] / equity.iloc[0] - 1.0
    print(f'β={beta:.1f}: {item["entries"]} entries; compound return {total_return:+.2%}')

In [ ]:
cmap = LinearSegmentedColormap.from_list('regime', ['#b91c1c', '#facc15', '#166534'])
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

betas = list(results_by_beta)
reference = results_by_beta[betas[0]]['regime']
image = np.ma.masked_invalid(np.vstack([results_by_beta[beta]['regime']['erc'].to_numpy() for beta in betas]))
fig, axis = plt.subplots(figsize=(16, 4.8), constrained_layout=True)
fig.suptitle(f'SBLK / BHP · {BASELINE_STRATEGY_NAME}', x=0.01, ha='left', fontweight='bold')
axis.imshow(image, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest', extent=[0, len(reference) - 1, len(betas) - 0.5, -0.5])
axis.set(title='Equity Regime Colour by fixed beta', xlabel='Trading time', ylabel='Fixed beta')
tick_positions = np.linspace(0, len(reference) - 1, 7, dtype=int)
axis.set_xticks(tick_positions, [reference.index[i].strftime('%Y-%m') for i in tick_positions])
axis.set_yticks(range(len(betas)), [f'β = {beta:.1f}' for beta in betas])
for row, beta in enumerate(betas):
    equity = results_by_beta[beta]['regime']['equity']
    total_return = equity.iloc[-1] / equity.iloc[0] - 1.0
    axis.text(1.01, 1.0 - (row + 0.5) / len(betas), f'{total_return:+.1%}', transform=axis.transAxes, va='center', ha='left', fontweight='bold')
axis.text(1.01, 1.04, 'Compound\ntotal return', transform=axis.transAxes, va='bottom', ha='left', fontweight='bold')
colourbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axis, pad=0.02)
colourbar.set_label('ERC: red = hostile, yellow = neutral / drawdown, green = favourable')
plt.show()

In [ ]:
# Daily companion: use the final common 5-minute bar of each US session.
def load_sblk_bhp_daily():
    query = """
        SELECT a.time_utc, a.price AS sblk, b.price AS bhp
        FROM us_market_prices a FORCE INDEX (idx_us_market_prices_ticker_time_price)
        JOIN us_market_prices b FORCE INDEX (idx_us_market_prices_ticker_time_price)
          ON b.ticker = 'BHP' AND b.time_utc = a.time_utc
        JOIN (
            SELECT DATE(time_utc) AS trading_date, MAX(time_utc) AS close_time
            FROM us_market_prices FORCE INDEX (idx_us_market_prices_ticker_time_price)
            WHERE ticker = 'SBLK'
            GROUP BY DATE(time_utc)
        ) sessions ON sessions.close_time = a.time_utc
        WHERE a.ticker = 'SBLK'
        ORDER BY a.time_utc
    """
    print('Loading daily SBLK/BHP closes from MySQL…', flush=True)
    conn = pymysql.connect(**DB_CONFIG, connect_timeout=10, read_timeout=60, write_timeout=60)
    try:
        data = pd.read_sql(query, conn)
    finally:
        conn.close()
    data['time_utc'] = pd.to_datetime(data['time_utc'], utc=True)
    return data.dropna().drop_duplicates('time_utc').reset_index(drop=True)

daily_bars = load_sblk_bhp_daily()
daily_results_by_beta = {}
for beta in BETA_GRID:
    daily_backtest, entry_count = base_strategy_equity(daily_bars, beta=beta, signal_window=20)
    daily_regime = equity_regime_colour(daily_backtest['equity'], growth_window=20, risk_window=60)
    daily_regime.index = daily_backtest['time_utc']
    daily_results_by_beta[float(beta)] = {'regime': daily_regime, 'entries': entry_count}

daily_betas = list(daily_results_by_beta)
daily_reference = daily_results_by_beta[daily_betas[0]]['regime']
daily_image = np.ma.masked_invalid(np.vstack([daily_results_by_beta[beta]['regime']['erc'].to_numpy() for beta in daily_betas]))
daily_fig, daily_axis = plt.subplots(figsize=(16, 4.8), constrained_layout=True)
daily_fig.suptitle(f'SBLK / BHP · Daily Bars · {BASELINE_STRATEGY_NAME}', x=0.01, ha='left', fontweight='bold')
daily_axis.imshow(daily_image, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest', extent=[0, len(daily_reference) - 1, len(daily_betas) - 0.5, -0.5])
daily_axis.set(title='Daily Equity Regime Colour by fixed beta', xlabel='Trading time', ylabel='Fixed beta')
daily_ticks = np.linspace(0, len(daily_reference) - 1, 7, dtype=int)
daily_axis.set_xticks(daily_ticks, [daily_reference.index[i].strftime('%Y-%m') for i in daily_ticks])
daily_axis.set_yticks(range(len(daily_betas)), [f'β = {beta:.1f}' for beta in daily_betas])
for row, beta in enumerate(daily_betas):
    equity = daily_results_by_beta[beta]['regime']['equity']
    total_return = equity.iloc[-1] / equity.iloc[0] - 1.0
    daily_axis.text(1.01, 1.0 - (row + 0.5) / len(daily_betas), f'{total_return:+.1%}', transform=daily_axis.transAxes, va='center', ha='left', fontweight='bold')
daily_axis.text(1.01, 1.04, 'Compound\ntotal return', transform=daily_axis.transAxes, va='bottom', ha='left', fontweight='bold')
daily_colourbar = daily_fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=daily_axis, pad=0.02)
daily_colourbar.set_label('ERC: red = hostile, yellow = neutral / drawdown, green = favourable')
print(f'{len(daily_bars):,} daily bars: {daily_bars.time_utc.iloc[0]} to {daily_bars.time_utc.iloc[-1]}')
plt.show()

## 5. 参数不是答案，而是研究假设

推荐从日频 $h=20,w=60,d_0=10\%$ 开始；5 分钟策略应把窗口换算为交易日数量，而不是随意沿用 20 根 bar。以下检验比寻找“最好看”的参数重要：

- **稳定性：** 改动 $h,w,d_0$ 一小段范围，绿色/红色分区是否仍大致一致？
- **前瞻性：** 用 $C_t$ 分箱后，检查下一段固定持有期的真实收益、波动与回撤；仅看同期 equity 没有预测价值。
- **样本外：** 参数和颜色阈值在训练集冻结，在后续区间评估。
- **跨策略公平性：** 对相同频率、相同交易成本假设、相同时间戳对齐的 equity 使用同一公式。
- **回撤语义：** 若策略设计上允许较深回撤，$d_0$ 应由其风险预算给出，而不是为了让图更绿。

## 6. 为成千上万 pair 自动设定 beta 扫描范围：市场 beta 是先验，不是协整答案

设日收益满足市场模型：

$$
r^Y_t = \alpha_Y + b_Y r^M_t + \varepsilon^Y_t
$$

$$
r^X_t = \alpha_X + b_X r^M_t + \varepsilon^X_t
$$

项目中的价差定义为 $S_t=Y_t-hX_t$。若 $Y_t,X_t$ 是**原始每股价格**，一个 long 1 share $Y$、short $h$ shares $X$ 的一阶市场 beta 中性条件是：

$$Y_t b_Y-hX_t b_X=0,\qquad h_M(t)=\frac{Y_t b_Y}{X_t b_X}. $$

若先把两条腿都按同一基期归一化为 $Y/Y_0$、$X/X_0$，才可省去价格比，得到 $h_M=b_Y/b_X$。若采用另一种记号 $S=Y+\beta X$，则 $\beta=-h_M$。因此“负的 $-0.53$”与本项目的正 hedge ratio $h=+0.53$ 可以描述同一件事，只是 spread 的符号记号不同。

以记忆中的 ADM/MOS 数字为**条件例子**：若某一冻结估计窗得到 $b_{ADM}=0.71,b_{MOS}=1.34$，归一化腿的市场中性中心是 $h_M=0.71/1.34\approx0.53$；以 $S=ADM+\beta MOS$ 表示时，系数是 $\beta\approx-0.53$。但若直接对原始 ADM/MOS 股价做 $ADM-h\,MOS$，还必须使用当期或预先冻结的参考价格比 $ADM/MOS$。

这不是协整 beta。市场 beta 控制的是两腿对共同市场收益的瞬时线性暴露；协整 hedge ratio $h_C$ 控制的是价格水平组合的长期平稳性。两者相近是有价值的经济一致性证据，明显不同则是一个应被记录和检验的信号，而不是强行把其中一个当作真值。

## 7. 可机器化的两阶段 beta 网格

对每一 pair，先在同一个、事前冻结的日线窗口估计：

1. $h_M$：上式的市场 beta 中性 hedge ratio；当 $|b_X|$ 很小、置信区间穿过 0、或价格缺失时标记为市场 beta 不可靠，不要做比值。
2. $h_C$：低成本的价格级候选，例如端点去趋势 OLS、最小方差 beta 或 Engle--Granger 第一步斜率；它仅是筛选中心，随后仍要做独立样本检验。

对于同号且正的股票 hedge ratio，可用对数对称网格避免大 beta 获得不成比例宽的搜索区：

$$\mathcal H_M=\{h_Me^{u_j}:u_j\in[-a,a]\},\qquad\mathcal H_C=\{h_Ce^{u_j}:u_j\in[-a,a]\}. $$

例如取 $a=\log(1.35)$、$u_j$ 等距 7 点，候选集合为 $\mathcal H=\operatorname{unique}(\mathcal H_M\cup\mathcal H_C)$，并限制到事前规定的合理仓位上限。这样每 pair 只需约 10--14 个候选，而不是从 0 到很大的 beta 盲扫。

如果 $h_M,h_C$ 符号不同、相差过大，或任一估计不可靠，机器不应自动扩大范围来“找到最好回测”；应把 pair 放入异常队列，改为只围绕可靠中心做窄网格，或留给研究复核。扫描评分也必须在训练窗生成候选、在后续测试窗比较 ERC、回报、回撤和交易数，不能在同一全样本上选最大收益。

## 8. 可继续发明的两个分支

1. **只关心变化率：** 使用 $G_t^{(h)}$，这是最透明的颜色定义。
2. **关心是否适合启用策略：** 使用 ERC 的 $C_t$，或进一步加入交易成本、换手率、尾部损失等惩罚项。

下一课可用真实 `pair_backtest_game` 的 equity 序列作输入，先画多条策略的 ERC 色带，再做一个严格的样本外检验：$C_t$ 为绿时，未来 5/20 个交易日的策略表现是否确实更好。若没有，这个指标仍可作为描述性仪表盘，但不应被解释成市场状态预测器。